In [1]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [2]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    # Function to evaluate the expectation value of the Hamiltonian for a given set of parameters
    # Inputs: parameter_values (ndarray), parameter values
    # Outputs: result (float), energy value

    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars =  list(value_dict.values())
    expectation_value = estimator.run(ansatz,qubit_op,pars).result().values
    return np.real(expectation_value)

In [3]:
from abc import ABCMeta, abstractmethod
import types
import warnings

class SkoBase(metaclass=ABCMeta):
    def register(self, operator_name, operator, *args, **kwargs):
        '''
        regeister udf to the class
        :param operator_name: string
        :param operator: a function, operator itself
        :param args: arg of operator
        :param kwargs: kwargs of operator
        :return:
        '''

        def operator_wapper(*wrapper_args):
            return operator(*(wrapper_args + args), **kwargs)

        setattr(self, operator_name, types.MethodType(operator_wapper, self))
        return self

    def fit(self, *args, **kwargs):
        warnings.warn('.fit() will be deprecated in the future. use .run() instead.'
                      , DeprecationWarning)
        return self.run(*args, **kwargs)


class Problem(object):
    pass


class SimulatedAnnealingBase(SkoBase):
    """
    DO SA(Simulated Annealing)

    Parameters
    ----------------
    func : function
        The func you want to do optimal
    n_dim : int
        number of variables of func
    x0 : array, shape is n_dim
        initial solution
    T_max :float
        initial temperature
    T_min : float
        end temperature

    Attributes
    ----------------------


    Examples
    -------------
    See https://github.com/guofei9987/scikit-opt/blob/master/examples/demo_sa.py
    """

    def __init__(self, func, x0, T_max=100, T_min=1e-7, L=300, max_stay_counter=150, **kwargs):
        assert T_max > T_min > 0, 'T_max > T_min > 0'

        self.func = func
        self.T_max = T_max  # initial temperature
        self.T_min = T_min  # end temperature
        self.L = int(L)  # num of iteration under every temperature（also called Long of Chain）
        # stop if best_y stay unchanged over max_stay_counter times (also called cooldown time)
        self.max_stay_counter = max_stay_counter

        self.n_dim = len(x0)

        self.best_x = np.array(x0)  # initial solution
        self.best_y = self.func(self.best_x)
        self.T = self.T_max
        self.iter_cycle = 0
        self.generation_best_X, self.generation_best_Y = [self.best_x], [self.best_y]
        # history reasons, will be deprecated
        self.best_x_history, self.best_y_history = self.generation_best_X, self.generation_best_Y

        # Initialize function evaluation counter
        self.func_evals = 1  # Counting the initial evaluation

    def get_new_x(self, x):
        u = np.random.uniform(-1, 1, size=self.n_dim)
        x_new = x + 20 * np.sign(u) * self.T * ((1 + 1.0 / self.T) ** np.abs(u) - 1.0)
        return x_new

    def cool_down(self):
        self.T = self.T * 0.7

    def isclose(self, a, b, rel_tol=1e-09, abs_tol=1e-30):
        return abs(a - b) <= max(rel_tol * max(abs(a), abs(b)), abs_tol)

    def run(self, qubits):
        x_current, y_current = self.best_x, self.best_y
        stay_counter = 0
        while np.abs(self.best_y + qubits - 1) >= 1e-1:
            for i in range(self.L):
                x_new = self.get_new_x(x_current)
                y_new = self.func(x_new)
                self.func_evals += 1  # Increment function evaluation counter

                # Metropolis
                df = y_new - y_current
                if df < 0 or np.exp(-df / self.T) > np.random.rand():
                    x_current, y_current = x_new, y_new
                    if y_new < self.best_y:
                        self.best_x, self.best_y = x_new, y_new

            self.iter_cycle += 1
            self.cool_down()
            self.generation_best_Y.append(self.best_y)
            self.generation_best_X.append(self.best_x)

            # if best_y stay for max_stay_counter times, stop iteration
            if self.isclose(self.best_y_history[-1], self.best_y_history[-2]):
                stay_counter += 1
            else:
                stay_counter = 0

            if self.T < self.T_min:
                stop_code = 'Cooled to final temperature'
                break
            if stay_counter > self.max_stay_counter:
                stop_code = 'Stay unchanged in the last {stay_counter} iterations'.format(stay_counter=stay_counter)
                break

        print("Total number of function evaluations:", self.func_evals)
        return self.best_x, self.best_y, self.func_evals

    fit = run


class SimulatedAnnealingValue(SimulatedAnnealingBase):
    """
    SA on real value function
    """

    def __init__(self, func, x0, T_max=100, T_min=1e-7, L=300, max_stay_counter=150, **kwargs):
        super().__init__(func, x0, T_max, T_min, L, max_stay_counter, **kwargs)
        lb, ub = kwargs.get('lb', None), kwargs.get('ub', None)

        if lb is not None and ub is not None:
            self.has_bounds = True
            self.lb, self.ub = np.array(lb) * np.ones(self.n_dim), np.array(ub) * np.ones(self.n_dim)
            assert self.n_dim == len(self.lb) == len(self.ub), 'dim == len(lb) == len(ub) is not True'
            assert np.all(self.ub > self.lb), 'upper-bound must be greater than lower-bound'
            self.hop = kwargs.get('hop', self.ub - self.lb)
        elif lb is None and ub is None:
            self.has_bounds = False
            self.hop = kwargs.get('hop', 10)
        else:
            raise ValueError('input parameter error: lb, ub both exist, or both not exist')
        self.hop = self.hop * np.ones(self.n_dim)


class SAFast(SimulatedAnnealingValue):
    """
    u ~ Uniform(0, 1, size = d)
    y = sgn(u - 0.5) * T * ((1 + 1/T)**abs(2*u - 1) - 1.0)

    xc = y * (upper - lower)
    x_new = x_old + xc

    c = n * exp(-n * quench)
    T_new = T0 * exp(-c * k**quench)
    """

    def __init__(self, func, x0, T_max=100, T_min=1e-7, L=300, max_stay_counter=150, **kwargs):
        super().__init__(func, x0, T_max, T_min, L, max_stay_counter, **kwargs)
        self.m, self.n, self.quench = kwargs.get('m', 1), kwargs.get('n', 1), kwargs.get('quench', 1)
        self.c = self.m * np.exp(-self.n * self.quench)

    def get_new_x(self, x):
        r = np.random.uniform(-1, 1, size=self.n_dim)
        xc = np.sign(r) * self.T * ((1 + 1.0 / self.T) ** np.abs(r) - 1.0)
        x_new = x + xc * self.hop
        if self.has_bounds:
            return np.clip(x_new, self.lb, self.ub)
        return x_new

    def cool_down(self):
        self.T = self.T_max * np.exp(-self.c * self.iter_cycle ** self.quench)

class SABoltzmann(SimulatedAnnealingValue):
    """
    std = minimum(sqrt(T) * ones(d), (upper - lower) / (3*learn_rate))
    y ~ Normal(0, std, size = d)
    x_new = x_old + learn_rate * y

    T_new = T0 / log(1 + k)
    """

    def __init__(self, func, x0, T_max=100, T_min=1e-7, L=300, max_stay_counter=150, **kwargs):
        super().__init__(func, x0, T_max, T_min, L, max_stay_counter, **kwargs)
        self.learn_rate = kwargs.get('learn_rate', 0.5)

    def get_new_x(self, x):
        a, b = np.sqrt(self.T), self.hop / 3.0 / self.learn_rate
        std = np.where(a < b, a, b)
        xc = np.random.normal(0, 1.0, size=self.n_dim)
        x_new = x + xc * std * self.learn_rate
        if self.has_bounds:
            return np.clip(x_new, self.lb, self.ub)
        return x_new

    def cool_down(self):
        self.T = self.T_max / np.log(self.iter_cycle + 1.0)


class SACauchy(SimulatedAnnealingValue):
    """
    u ~ Uniform(-pi/2, pi/2, size=d)
    xc = learn_rate * T * tan(u)
    x_new = x_old + xc

    T_new = T0 / (1 + k)
    """

    def __init__(self, func, x0, T_max=100, T_min=1e-7, L=300, max_stay_counter=150, **kwargs):
        super().__init__(func, x0, T_max, T_min, L, max_stay_counter, **kwargs)
        self.learn_rate = kwargs.get('learn_rate', 0.5)

    def get_new_x(self, x):
        u = np.random.uniform(-np.pi / 2, np.pi / 2, size=self.n_dim)
        xc = self.learn_rate * self.T * np.tan(u)
        x_new = x + xc
        if self.has_bounds:
            return np.clip(x_new, self.lb, self.ub)
        return x_new

    def cool_down(self):
        self.T = self.T_max / (1 + self.iter_cycle)


In [ ]:
min_qubits = 3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )

    sa_fast = SAFast(func=evaluate_expectation, x0=np.zeros(k*4), T_max=1, T_min=1e-9, q=0.99, L=k*100, max_stay_counter=1000)
    sa_fast.run(k)
    print('Fast Simulated Annealing: best_x is ', sa_fast.best_x, 'best_y is ', sa_fast.best_y)
    print("Number of qubits: ", k, "Number of parameters: ", k*4)

ansatz_num_parameters= 12
Total number of function evaluations: 7201
Fast Simulated Annealing: best_x is  [  57.33406467  -90.31456817  126.54982849   13.64596815  203.96197035
  -58.48858507 -220.19370828   23.502463    -53.60849012  -62.32789756
  -32.9197227  -129.50906069] best_y is  [-1.90625]
Number of qubits:  3 Number of parameters:  12
ansatz_num_parameters= 16
Total number of function evaluations: 10401
Fast Simulated Annealing: best_x is  [ 131.2940019  -157.76892067 -126.45860207 -241.87753579 -122.63472633
 -219.57983612 -138.65196616   61.58863108 -205.91590293   29.82685179
   -7.99352466 -184.24460121  -29.60472821   55.24695242  -14.07916004
 -118.89620314] best_y is  [-2.90625]
Number of qubits:  4 Number of parameters:  16
ansatz_num_parameters= 20
Total number of function evaluations: 23001
Fast Simulated Annealing: best_x is  [-195.58688478 -419.96924853  197.19340872   -8.75280168   36.8139071
  -32.87963819  -18.44349218  248.68781063  -74.00889607   10.25253424


In [ ]:
min_qubits = 3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )

    sa_boltz = SABoltzmann(func=evaluate_expectation, x0=np.zeros(k*4), T_max=1, T_min=1e-9, q=0.99, L=k*100, max_stay_counter=1000)
    sa_boltz.run(k)
    print('Fast Simulated Annealing: best_x is ', sa_boltz.best_x, 'best_y is ', sa_boltz.best_y)
    print("Number of qubits: ", k, "Number of parameters: ", k*4)

ansatz_num_parameters= 12
Total number of function evaluations: 4801
Fast Simulated Annealing: best_x is  [-49.91505095 -21.93647031 -11.46351228   3.92474924   6.38742273
   1.76308488 -22.68637712  20.28109066 -33.13998695 -10.00984319
 -17.3944042    6.05241511] best_y is  [-1.90625]
Number of qubits:  3 Number of parameters:  12
ansatz_num_parameters= 16
Total number of function evaluations: 35601
Fast Simulated Annealing: best_x is  [ 26.12314317  -3.72364981   0.85226984   2.38103512   3.01808827
 -24.45264474  57.89909958 -44.7557341   -9.61560209  29.77313811
   2.39819975  29.58256223 -20.52615541 -11.01425139 -18.9570723
 -36.06454779] best_y is  [-2.96875]
Number of qubits:  4 Number of parameters:  16
ansatz_num_parameters= 20
Total number of function evaluations: 50501
Fast Simulated Annealing: best_x is  [ 42.46788084  11.98697471 -18.56094413  -0.61095951 -45.52411833
 -30.82595577  19.14250463  71.15507373  35.11484727 -28.27378495
  11.59649075   1.32212079   8.6115694

KeyboardInterrupt: 

In [ ]:
min_qubits = 3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )

    sa_boltz = SACauchy(func=evaluate_expectation, x0=np.zeros(k*4), T_max=1, T_min=1e-9, q=0.99, L=k*50, max_stay_counter=500)
    sa_boltz.run(k)
    print('Cauchy Simulated Annealing: best_y is ', sa_boltz.best_y)
    print("Number of qubits: ", k, "Number of parameters: ", k*4)

ansatz_num_parameters= 12
Total number of function evaluations: 751
Cauchy Simulated Annealing: best_y is  [-1.9375]
Number of qubits:  3 Number of parameters:  12
ansatz_num_parameters= 16
Total number of function evaluations: 1801
Cauchy Simulated Annealing: best_y is  [-2.90625]
Number of qubits:  4 Number of parameters:  16
ansatz_num_parameters= 20
Total number of function evaluations: 5251
Cauchy Simulated Annealing: best_y is  [-3.9375]
Number of qubits:  5 Number of parameters:  20
ansatz_num_parameters= 24
Total number of function evaluations: 4201
Cauchy Simulated Annealing: best_y is  [-4.9375]
Number of qubits:  6 Number of parameters:  24
ansatz_num_parameters= 28
Total number of function evaluations: 14351
Cauchy Simulated Annealing: best_y is  [-5.90625]
Number of qubits:  7 Number of parameters:  28
ansatz_num_parameters= 32
Total number of function evaluations: 8001
Cauchy Simulated Annealing: best_y is  [-6.90625]
Number of qubits:  8 Number of parameters:  32
ansatz_